In [ ]:
# Compatible Google Colab — aucune installation requise


# Module 08 — Console et journalisation

## Objectifs
- Concevoir une boucle parser → valider → exécuter
- Lire et valider des saisies utilisateur sans regex
- Gérer deux joueurs (J1 Drones / J2 Tempêtes)
- Écrire un journal de partie (`partie.log`) et un bilan (`resultats.txt`)


## Concepts clés

### Boucle parser → valider → exécuter
```python
saisie = input('Destination : ').strip().upper()  # 1. lire
cible = position_depuis_chaine(saisie)            # 2. parser
if cible is None:
    print('Format invalide')
    continue
ok, raison = valider_mouvement(drone, cible)      # 3. valider
if not ok:
    print(raison)
    continue
executer_mouvement(drone, cible)                   # 4. exécuter
```

### Parser sans regex — position_depuis_chaine
```python
LETTRES = 'ABCDEFGHIJ'
def position_depuis_chaine(texte):
    texte = texte.strip().upper()
    if len(texte) < 2:
        return None
    lettre = texte[0]
    if lettre not in LETTRES:
        return None
    try:
        lig = int(texte[1:]) - 1   # 1-based → 0-based
    except ValueError:
        return None
    col = LETTRES.index(lettre)
    return (col, lig)
```

### Journalisation avec open() + with
```python
def enregistrer_log(chemin, ligne):
    with open(chemin, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')
```


In [ ]:
LETTRES = 'ABCDEFGHIJ'
TAILLE  = 5

def position_depuis_chaine(texte):
    """Convertit 'B3' → (1, 2). Retourne None si invalide."""
    texte = texte.strip().upper()
    if len(texte) < 2:
        return None
    lettre = texte[0]
    if lettre not in LETTRES:
        return None
    try:
        lig = int(texte[1:]) - 1
    except ValueError:
        return None
    col = LETTRES.index(lettre)
    if 0 <= col < TAILLE and 0 <= lig < TAILLE:
        return (col, lig)
    return None

# Tests
assert position_depuis_chaine('B3') == (1, 2)
assert position_depuis_chaine('A1') == (0, 0)
assert position_depuis_chaine('Z9') is None   # lettre hors grille
assert position_depuis_chaine('abc') is None  # format invalide
assert position_depuis_chaine('B0') is None   # ligne hors grille (0 non valide)
print('Parser OK ✓')

## Lien avec Drone Rescue

Dans `jeu/console.py`, la boucle `_phase_drones()` suit exactement le schéma parser→valider→exécuter.
Dans `jeu/logger.py`, `enregistrer_log()` utilise `open()` en mode `'a'` (append) pour ne pas effacer l'historique.
Le fichier `resultats.txt` est créé séparément par `sauvegarder_resultats()` — contrainte du sujet.


In [ ]:
import io, sys

# Simuler la journalisation sans créer de fichier réel
journal = []  # on simule le fichier par une liste

def enregistrer_log_sim(ligne):
    """Version simulation : ajoute au journal en mémoire."""
    journal.append(ligne)

def formater_log(tour, identifiant, depart, arrivee, evenement=''):
    """Formate une ligne de log condensée."""
    dep = f"{LETTRES[depart[0]]}{depart[1]+1}"
    arr = f"{LETTRES[arrivee[0]]}{arrivee[1]+1}"
    ligne = f"T{tour:02d}  {identifiant:3s}  {dep}→{arr}"
    if evenement:
        ligne += f"  {evenement}"
    return ligne

# Test
ligne = formater_log(4, 'D1', (1, 2), (2, 3), 'PRISE S2')
enregistrer_log_sim(ligne)
print(journal[0])  # T04  D1   B3→C4  PRISE S2

## À toi de jouer !

**Exercice A** : Écris `simuler_saisie(texte, drones, etat)` qui simule une saisie utilisateur, parse la destination et retourne le résultat de validation.

**Exercice B** : Écris `sauvegarder_resultats_sim(etat)` qui retourne une chaîne multi-lignes avec l'issue (VICTOIRE/DÉFAITE), le score, les tours et le détail des drones — sans écrire de fichier réel.


In [ ]:
def simuler_saisie(texte, drone, zones_x=set()):
    """Parse texte, retourne (cible, ok, raison) sans exécuter."""
    pass

# Test
# d = {'id': 'D1', 'col': 0, 'lig': 0, 'batterie': 10, 'bloque': 0, 'hors_service': False, 'survivant': None}
# cible, ok, raison = simuler_saisie('B2', d)
# assert ok
# cible, ok, raison = simuler_saisie('ZZZ', d)
# assert not ok

In [ ]:
# SOLUTION
# def simuler_saisie(texte, drone, zones_x=set()):
#     cible = position_depuis_chaine(texte)
#     if cible is None:
#         return None, False, 'Format invalide'
#     if drone['hors_service']:
#         return cible, False, f"{drone['id']} hors service"
#     if drone['bloque'] > 0:
#         return cible, False, 'Drone bloqué'
#     dist = max(abs(cible[0]-drone['col']), abs(cible[1]-drone['lig']))
#     if dist > 1:
#         return cible, False, 'Trop loin'
#     if drone['batterie'] <= 0:
#         return cible, False, 'Plus de batterie'
#     return cible, True, ''